# CharWriteViT on Kaggle
Stage 1 uses only `crawford/emnist` plus a dataset of licensed handwriting fonts. IAM is attached later for OCR and generator training.

In [ ]:
REPOSITORY = 'https://github.com/YOUR_USER/YOUR_REPO.git'
!git clone $REPOSITORY /kaggle/working/CharWriteViT
%cd /kaggle/working/CharWriteViT
!pip install -q -e .

In [ ]:
# Attach crawford/emnist and licensed fonts. Edit configs/kaggle.yaml if paths differ.
!find /kaggle/input -maxdepth 4 -type f | head -80
!python -m charwritevit.cli.prepare_char_data --config configs/kaggle.yaml

In [ ]:
# Smoke test before spending a full Kaggle session.
from pathlib import Path
import subprocess, sys
from IPython.display import display, Image

PROJECT = Path('/kaggle/working/CharWriteViT')
SMOKE_OUTPUT = PROJECT / 'outputs' / 'classifier-smoke'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_classifier',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(SMOKE_OUTPUT), '--epochs', '1',
                '--max-steps', '20'], cwd=PROJECT, check=True)

for artifact in [SMOKE_OUTPUT / 'classifier_curves.png',
                 SMOKE_OUTPUT / 'preview' / 'epoch_000.png']:
    if not artifact.is_file():
        raise RuntimeError(f'Training exited successfully but did not create {artifact}')
    display(Image(filename=str(artifact)))

In [ ]:
# Full classifier training. Save best.pt as a Kaggle output before continuing.
CLASSIFIER_OUTPUT = PROJECT / 'outputs' / 'classifier'
subprocess.run([sys.executable, '-m', 'charwritevit.cli.train_classifier',
                '--config', str(PROJECT / 'configs/kaggle.yaml'),
                '--output', str(CLASSIFIER_OUTPUT)], cwd=PROJECT, check=True)
curve = CLASSIFIER_OUTPUT / 'classifier_curves.png'
if not curve.is_file():
    raise RuntimeError(f'Training exited successfully but did not create {curve}')
display(Image(filename=str(curve)))

In [ ]:
# Later stages: run only after inspecting and saving classifier/best.pt.
!python -m charwritevit.cli.prepare_iam --config configs/kaggle.yaml
!python -m charwritevit.cli.train_synthetic --config configs/kaggle.yaml --classifier outputs/classifier/best.pt --output outputs/synthetic
!python -m charwritevit.cli.train_ocr --config configs/kaggle.yaml --output outputs/ocr
!python -m charwritevit.cli.train_iam --config configs/kaggle.yaml --classifier outputs/classifier/best.pt --ocr outputs/ocr/latest.pt --synthetic-checkpoint outputs/synthetic/latest.pt --warmup-epochs 5 --output outputs/iam

In [ ]:
# Archive checkpoints so the notebook Output can be saved as a Kaggle Dataset.
!tar -czf /kaggle/working/charwritevit-checkpoints.tgz outputs configs/kaggle.yaml